In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed endpoints: locate the finite forward response by layer

Select a GPU runtime and **Run all**. This diagnostic reuses the starting POINT3 terminal and both COMPOSITE/WORST_ONLY endpoints from run 20261002T053509124852Z. It does not update any terminal or choose a winning endpoint.

Each of the three saved terminals is decoded once by the same native FP32 Wan VAE. FLOAT means the existing decoded/2+0.5 mapping and [0,1] clamp, without integer quantization. RGB8 adds only the existing uint8 quantization and division by 255. Both are separately mode-encoded and normalized by the same VAE. Saved float RGB, RGB8, normalized tensors and full blind readouts remain available for independent recomputation. This does not separately isolate clamp effects.

Six new observations, twelve path and twelve payload reads, twenty-four message comparisons and 2088 valid candidate costs are fixed. Every read retains all 174 valid candidates at fixed phase 0 and R44. Three native decodes and six native encodes; zero writer updates, gradient/VJP calls, generation or new color/codec conversions. The six old raw420/raw444 observations and twelve old raw readouts are hash-verified references, not new evidence. Both keys and messages are retained. A point or layer failure stays in the denominator while independent points/layers continue.

Compare each arm against the baseline of the SAME layer: minimum wrong-minus-true gap, rank, global-worst term, local mean and composite loss. Then compare the signed finite responses between FLOAT and RGB8, and between RGB8 and each old raw channel. Raw444 is a parallel contrast, not a stage after raw420. The previous gradient prediction belongs to the original raw420 identity-STE computation; it is not a true gradient of the new FLOAT or RGB8 layer. A sign reversal localizes an observed discrepancy and does not uniquely identify its cause.

Each new RGB8 raster is compared by byte hash with the actual input raster recorded for both old channels. A mismatch is reported and limits cross-run attribution; it does not discard the new observations or require a matching GPU/environment. Current Python with the established dependency repair is used. No GPU model requirement, venv or ensurepip.

This notebook is a user-run diagnostic with static/CPU validation only; the complete pretrained VAE run is pending user execution. Results persist under MyDrive/Video-WM/Zero-Mean-Fixed-Layer-Forward-V1/<timestamp>/fixed_reference/result.json. Recovery at these fixed saved terminals does not establish unknown-phase synchronization, path-aligned payload closure, generation-time trajectory embedding, MP4 robustness or population FPR.


In [ ]:
SOURCE_SHA = '6d07159bad76ce7f5de43bd70755df8a96dd6eba'
from pathlib import Path
import datetime,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Fixed-Layer-Forward-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-FIXED-LAYER-FORWARD-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
INPUT_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Raw420-Objective-Ablation-V1/20261002T053509124852Z/fixed_reference')
REFERENCE_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Raw420-Margin-Continue-V2/20261002T041230450908Z/fixed_reference')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator={'fixed_terminals': 3, 'new_observations': 6, 'path_reads': 12, 'payload_reads': 12, 'message_evaluations': 24, 'valid_costs': 2088, 'catalog_slots': 46980, 'structural_exclusions': 44892, 'reference_observations': 6, 'reference_raw': 12, 'updates': 0},observations={o:{'status':'NOT_RUN_SETUP'} for o in ['BASELINE_FLOAT', 'BASELINE_RGB8', 'COMPOSITE_FLOAT', 'COMPOSITE_RGB8', 'WORST_ONLY_FLOAT', 'WORST_ONLY_RGB8']})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Full writer terminal:',INPUT_ROOT,'Saved references:',REFERENCE_ROOT,'Output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import AutoencoderKLWan; from experiments.wan_state_clock.zero_mean_c1_yuv444_no_h264_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.zero_mean_fixed_layer_forward_run','--output',str(RUN_OUTPUT),'--input-root',str(INPUT_ROOT),'--reference-root',str(REFERENCE_ROOT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=setup['fixed_denominator']:raise RuntimeError('result source/denominator mismatch')
print('Status:',result['status'],'Counts:',result.get('counts'))
print('Calls:',result['calls'])
print('Reference alignment:',result.get('reference_comparison_alignment'))
for name,row in result['replay'].items():print('Replay:',name,'exact=',row['exact_for_both_references'])
for name,row in result['surrogate_predictions'].items():print('Stored raw420 prediction:',name,row)
for name,row in result['comparisons'].items():print('Comparison:',name,row)
for name,row in result['layer_increments'].items():print('Layer response difference:',name,row)
for name,row in result['posthoc'].items():print(name,row)
for name,row in result['message_evaluations'].items():print(name,row)
for name,row in result['reference_observations'].items():
    print('Reference:',name,row['posthoc'])
    print('Reference messages:',name,row['messages'])
for name,row in result['observations'].items():
    loss=row.get('writer_loss',{})
    print('Observation:',name,row['status'],{k:loss.get(k) for k in ('loss','global_term','local_mean','local_active')})
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
